In [3]:
import ssl
import pandas as pd
import base64
from pyVim.connect import SmartConnect,Disconnect
from pyVmomi import vim

ssl_context = ssl.create_default_context(purpose=ssl.Purpose.CLIENT_AUTH)
ssl_context.verify_mode = ssl.CERT_NONE


loginUser=base64.b64decode('YXNfanp3eGZnQG5hbS5jb3JwLmdtLmNvbQ==').decode()
#loginpwd=base64.b64decode('dUJTWFp6NXdGUFlIZ3Bmaw==').decode()
loginpwd="Wc0MYgnKPd7sNHEq"

In [4]:
def get_vm(vm_name, vcs):
    vm=None
    vmname=None
    vmpower=None
    si = SmartConnect(host=vcs, user=loginUser, pwd=loginpwd,sslContext=ssl_context)
    vm_list = si.content.viewManager.CreateContainerView(si.content.rootFolder, [vim.VirtualMachine], True).view

    for vm in vm_list:
        if vm.name == vm_name:
            #outputvm = vcs[0],":",vm.name,":",vm.summary.guest.guestFullName,":",vm.summary.guest.guestId,":",vm.summary.guest.ipAddress,":",vm.summary.runtime.powerState
            vmname = vm.name#.runtime.powerState #name
            vmpower = vm.summary#.runtime#.powerState
         
    return vmname#,vmpower

In [5]:
si = SmartConnect(host=vcs, user=loginUser, pwd=loginpwd,sslContext=ssl_context)
vm_list = si.content.viewManager.CreateContainerView(si.content.rootFolder, [vim.VirtualMachine], True).view
print(type(vm_list))

NameError: name 'vcs' is not defined

In [6]:
vcs="dcwipvmvcs072.nam.corp.gm.com"
vmname='dcwipvmosp012'
out=get_vm(vmname,vcs)
print(out)

None


In [8]:
vsphere_client.vcenter.VM.list()

NameError: name 'vsphere_client' is not defined

### NEW more server a Vcenter
## kodu gelistir.
Flask kur. SR'in excelini SR numarasi iel upload et. sonra hostname stunundaki vmleri sira ile alip Vcenterlarini bul. vcenter stununu sort et. ve ayni vcenter daki sunucular icin statusu al. ciktisi pandastan excel e aktar ve asagidaki gibi bir cikti olsun. cikti adi da SRnumarasi_ServerDecom.csv


DateTime(ET) | Vcenter name | Vm_name | VM_State

In [1]:
import pandas as pd

def find_vcenter(vm_name):
    column_name = 'VM' 
    df = pd.read_csv('https://dcmipavvcs007.nam.corp.gm.com/vcsaupdate/vcupdaterepo/6.7U3_Patch/VM.csv')
    filtered_df = df[df[column_name].str.lower() == vm_name.lower()]
    if not filtered_df.empty:
        specific_column_value = filtered_df['vCenter'].iloc[0]
        #print("vCenter:", specific_column_value)
        return specific_column_value
    else:
        #print("vCenter not found")
        return "NoVcenter"

In [2]:
print(find_vcenter("dcmiddbhan801"))

NoVcenter


In [1]:
from pyVim.connect import SmartConnect, Disconnect
from pyVmomi import vim
import ssl
import time
import base64
from datetime import datetime
import pytz
import pandas as pd

def get_current_et_time():
    # Eastern Time Zone'u tanımla
    et_zone = pytz.timezone('America/New_York')
    
    # Şu anki zamanı al ve ET zaman dilimine göre ayarla
    et_time = datetime.now(et_zone)
    
    # Zamanı formatlayarak döndür
    return et_time.strftime('%Y-%m-%d %H:%M:%S')
  
#def connect_to_vcenter(host, user, pwd):
#   context = ssl._create_unverified_context()
#   si = SmartConnect(host=host, user=user, pwd=pwd, sslContext=context)
#   return si

def connect_to_vcenter(host, user, pwd):
    context = ssl._create_unverified_context()
    try:
        si = SmartConnect(host=host, user=user, pwd=pwd, sslContext=context)
        # Test the connection by retrieving some content
        content = si.RetrieveContent()
        if content:
            return si
        else:
            return None
    except Exception as e:
        print(f"Failed to connect to Vcenter: {e}")
        return None


def get_vm_by_name(si, vm_name):
    content = si.RetrieveContent()
    obj_view = content.viewManager.CreateContainerView(content.rootFolder, [vim.VirtualMachine], True)
    vm_list = obj_view.view
    obj_view.Destroy()
    for vm in vm_list:
        if vm.name.lower().split(" ")[0] == vm_name.lower():
            print ("====:"+str(vm.summary.runtime.powerState))
            return vm
            
    return None

def get_vm_status(vm,vcenter_host):
    summary = vm.summary
    print(f"Vcenter Info Vcebter Name : {vcenter_host}")
    print(f"Vcenter Info VM Obj Name  : {vm}")
    print(f"Vcenter Info VM Name      : {vm.name}")
    print(f"Vcenter Info VM State     : {summary.runtime.powerState}")
    
def process_vm(vm_name, si,vcenter_host):
    vm = get_vm_by_name(si, vm_name)
    if vm is None:
        print(f"VM {vm_name} not found.")
        return
    
    # Mevcut durumu göster
    get_vm_status(vm,vcenter_host)
    
def main():
    vcenter_host = "dcwipvmvcs072.nam.corp.gm.com"
    vcenter_user = base64.b64decode('YXNfanp3eGZnQG5hbS5jb3JwLmdtLmNvbQ==').decode()
    vcenter_password = "Wc0MYgnKPd7sNHEq" #base64.b64decode('dUJTWFp6NXdGUFlIZ3Bmaw==').decode()

    vm_names = [
"dcwipvmosp011",
"dcwipvmosp012",
"dcwipvmosp013",
"dcwipvmosp014",
"dcwipvmosp015",
"dcwipvmosp016",
"dcwipvmosp017",
"dcwipvmosp018",
"dcwipvmosp019",
"dcwipvmosp020",
"dcwipvmosp021",
"dcwipvmosp022",
"dcwipvmosp023",
"dcwipvmosp024",
"dcwipvmosp025",
"dcwipvmosp026",
"dcwipvmosp027",
"dcwipvmosp028",
"dcwipvmosp029",
"dcwipvmosp030"
       
    ]  # İşlem yapılacak VM isimleri listesi


    si = connect_to_vcenter(vcenter_host, vcenter_user, vcenter_password)
    
    try:
        current_et_time = get_current_et_time()
        print("Getting VMs INFO Date and Time - ET timezone:\n",current_et_time)
        print("="*50)
        for vm_name in vm_names:
            process_vm(vm_name, si,vcenter_host)
            print("="*50)
    
    finally:
        Disconnect(si)

if __name__ == "__main__":
    main()


Getting VMs INFO Date and Time - ET timezone:
 2024-11-13 06:45:16
====:poweredOff
Vcenter Info Vcebter Name : dcwipvmvcs072.nam.corp.gm.com
Vcenter Info VM Obj Name  : 'vim.VirtualMachine:vm-2251903'
Vcenter Info VM Name      : dcwipvmosp011 - control01
Vcenter Info VM State     : poweredOff
====:poweredOff
Vcenter Info Vcebter Name : dcwipvmvcs072.nam.corp.gm.com
Vcenter Info VM Obj Name  : 'vim.VirtualMachine:vm-2251904'
Vcenter Info VM Name      : dcwipvmosp012 - control02
Vcenter Info VM State     : poweredOff
====:poweredOff
Vcenter Info Vcebter Name : dcwipvmvcs072.nam.corp.gm.com
Vcenter Info VM Obj Name  : 'vim.VirtualMachine:vm-2262676'
Vcenter Info VM Name      : dcwipvmosp013 - control03
Vcenter Info VM State     : poweredOff
====:poweredOff
Vcenter Info Vcebter Name : dcwipvmvcs072.nam.corp.gm.com
Vcenter Info VM Obj Name  : 'vim.VirtualMachine:vm-2262678'
Vcenter Info VM Name      : dcwipvmosp014 - compute01
Vcenter Info VM State     : poweredOff
====:poweredOff
Vcenter I

In [19]:
import pandas as pd
import io

def find_hmc(vm_name):
    column_name = 'Name'
    my_url = "https://hmcscanner.gm.com/hmcdata/HMC_report.csv"
    proxies = {
        "http": "",
        "https": "",
              }
    s = requests.get(my_url, proxies=proxies).text
    df = pd.read_csv(io.StringIO(s))
    #df = pd.read_csv('https://hmcscanner.gm.com/hmcdata/HMC_report.csv')
    filtered_df = df[df[column_name].str.lower() == vm_name.lower()]
    if not filtered_df.empty:
        specific_column_value = filtered_df['HMC1'].iloc[0]
        #print("vCenter:", specific_column_value)
        return specific_column_value
    else:
        #print("vCenter not found")
        return "NoHMC"

In [23]:
print(find_hmc('dcmiddbhan261'))

dcmidvmhmc001.edc.nam.gm.com


In [21]:
import io
my_url = "https://hmcscanner.gm.com/hmcdata/HMC_report.csv"
proxies = {
  "http": "",
  "https": "",
}
s = requests.get(my_url, proxies=proxies).text

df = pd.read_csv(io.StringIO(s))

In [22]:
df

,Name,Status,OSVersion,ManagedSystemName,ManagedSystemSerial,HMC1,HMC2
0,dcwievmpe7007,NotActivated,Linux/SLES5.3.18-150300.59.115-defa15-SP315-SP3,DCWIPPHLNX001-SN78084A8,78084A8,dcwipphhmc005.edc.nam.gm.com,dcwipphhmc004.edc.nam.gm.com
1,dcwipdbhan111,NotActivated,Linux/SLES5.3.18-150300.59.174-defa15-SP315-SP3,DCWIPPHLNX001-SN78084A8,78084A8,dcwipphhmc005.edc.nam.gm.com,dcwipphhmc004.edc.nam.gm.com
2,dcwipvmpe7004,NotActivated,Linux/SLES5.3.18-150300.59.153-defa15-SP315-SP3,DCWIPPHLNX001-SN78084A8,78084A8,dcwipphhmc005.edc.nam.gm.com,dcwipphhmc004.edc.nam.gm.com
3,dcwipvmpe7003,NotActivated,Linux/SLES5.3.18-150300.59.118-defa15-SP315-SP3,DCWIPPHLNX001-SN78084A8,78084A8,dcwipphhmc005.edc.nam.gm.com,dcwipphhmc004.edc.nam.gm.com
4,dcwipvmpe7002,NotActivated,Linux/SLES5.3.18-150300.59.118-defa15-SP315-SP3,DCWIPPHLNX001-SN78084A8,78084A8,dcwipphhmc005.edc.nam.gm.com,dcwipphhmc004.edc.nam.gm.com
...,...,...,...,...,...,...,...
760,dcmirvmsle003,Running,Linux/SLES5.3.18-150300.59.158-defa15-SP315-SP3,DCMIRPHPWR002-SN7837A68,7837A68,dcmirphhmc001.rls.mpg.nam.gm.com,dcmirphhmc001.rls.mpg.nam.gm.com
761,dcmilvmvio012,Running,VIOS3.1.4.21,DCMIRPHPWR002-SN7837A68,7837A68,dcmirphhmc001.rls.mpg.nam.gm.com,dcmirphhmc001.rls.mpg.nam.gm.com
762,dcmilvmvio011,Running,VIOS3.1.4.21,DCMIRPHPWR002-SN7837A68,7837A68,dcmirphhmc001.rls.mpg.nam.gm.com,dcmirphhmc001.rls.mpg.nam.gm.com
763,dcmilvmvio016,Running,VIOS3.1.4.21,DCMIRPHPWR003-SN7896AD8,7896AD8,dcmirphhmc001.rls.mpg.nam.gm.com,dcmirphhmc001.rls.mpg.nam.gm.com
